# sLLM 파인튜닝 — 작은 공개 LLM을 내 데이터로 학습시키기

**군-학 연계 프로그램 범용 교육 6차시** · 카카오의 공개 모델 Kanana 1.5 (2.1B)에 민트상사(가상 회사) 고객센터 응대 200쌍과 사내 규정 Q&A 100쌍을 학습시킵니다.

## 사용 규칙 3가지
1. **위에서 아래로**, 셀 왼쪽의 ▶ 버튼을 하나씩 누릅니다. 셀이 끝나면 왼쪽 `[ ]` 안에 숫자가 생깁니다.
2. 바꿔 봐도 되는 줄은 `# <-- 바꿔 보세요` 표시가 있는 줄뿐입니다.
3. 빨간 글씨(에러)가 나오면 그대로 두고 손을 듭니다. 대부분 "셀 1을 안 돌림", "GPU를 안 켬", "순서를 건너뜀"입니다.

**오늘은 GPU가 필수입니다.** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**을 먼저 합니다. 셀 1이 GPU 이름을 알려 줍니다.

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 1 | 준비(도구 설치 · 모델 내려받기) | 약 5분 |
| 2 | 학습 전 — 시험 질문 10개 | 약 1.5분 |
| 3 | 데이터 300쌍 보기 | 바로 |
| 4 | 내 예시 추가 · 학습용으로 조립 | 바로 |
| 5 | 학습(LoRA) | 약 4분 |
| 6 | 학습 후 — 같은 질문 10개, 전/후 비교 | 약 1분 |
| 7 | 모르는 것을 물으면(환각) | 약 30초 |
| 8 | 파인튜닝 vs RAG(3차시 방식) | 약 1.5분 |
| 9 | 저장 · 다시 불러오기 · 내려받기 | 약 1분 |
| 10 | (선택) 내 상담 AI와 대화 | 바로 |

## 셀 1. 준비 — 도구를 설치하고 공개 모델을 내려받습니다
▶를 누르고 "준비 완료"가 나올 때까지 기다립니다(약 5분). 중간의 회색 글씨(경고)는 무시합니다.

In [ ]:
# 셀 1. 준비 — 도구 설치(약 3분) + 모델 내려받기(약 1분). "준비 완료"가 나올 때까지 기다립니다.
import subprocess, sys, os, time, json, re, gc, shutil
t0 = time.time()

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# unsloth 공식 Colab 설치 순서(2026-10 기준). 회색 경고는 무시합니다.
pip("sentencepiece", "protobuf", "datasets==4.3.0", "huggingface_hub>=0.34.0", "hf_transfer", "sentence-transformers")
pip("--no-deps", "unsloth_zoo", "bitsandbytes", "accelerate", "peft", "trl", "triton", "unsloth")
subprocess.run(["unsloth", "install-kernels"], check=False)
pip("--no-deps", "--upgrade", "torchao>=0.16.0")
pip("transformers==4.57.6")
pip("--no-deps", "trl==0.22.2")
print(f"설치 끝 ({(time.time() - t0) / 60:.1f}분)")

import torch
assert torch.cuda.is_available(), "GPU 없음 — 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU > 저장] 뒤 이 셀을 다시 실행하세요."
print("GPU:", torch.cuda.get_device_name(0))

from unsloth import FastLanguageModel
MODEL_ID = "kakaocorp/kanana-1.5-2.1b-instruct-2505"   # 카카오 공개 모델 (Apache 2.0, 한국어)
MAX_LEN = 1024                                          # 한 쌍(질문+답)의 최대 길이(토큰)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=False)
n_params = sum(p.numel() for p in model.parameters())
print(f"모델: {MODEL_ID}  다이얼(파라미터) {n_params / 1e9:.2f}B개")

import contextlib

def ask(question, max_new_tokens=220, adapter=True):
    """질문 한 개를 모델에 넣고 답 글자만 돌려준다. 같은 질문이면 같은 답이 나오게 무작위성은 끈다.
    adapter=False 면 보조 다이얼(LoRA)을 잠시 떼고 원래 공개 모델 그대로 답한다 (학습 뒤에만 의미 있음)."""
    messages = [{"role": "user", "content": question}]
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                        return_dict=True, return_tensors="pt").to("cuda")
    off = model.disable_adapter() if (not adapter and hasattr(model, "disable_adapter")) else contextlib.nullcontext()
    with torch.no_grad(), off:
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05,
                             use_cache=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
print("시험 질문 →", ask("한 줄로 자기소개 해 줘.", 60))
print(f"준비 완료 ({(time.time() - t0) / 60:.1f}분)")

## 셀 2. 학습 전 — 시험 질문 10개를 공개 모델 그대로에 물어봅니다
고객 문의 5개(말투·형식을 보는 질문)와 사내 규정 5개(숫자를 아는지 보는 질문)입니다. 답을 **셀 6에서 학습 후 답과 나란히** 비교합니다.

In [ ]:
# 셀 2. 학습 전 — 오늘의 시험 질문 10개를 공개 모델 그대로에 물어봅니다 (약 1.5분)
TEST_STYLE = [  # 고객 문의 5개 — 말투·형식을 보는 질문 (학습 데이터와 같은 주제, 다른 표현)
    "복합기 렌탈하다가 고장 나면 어떻게 되나요?",
    "주문한 의자가 아직 안 왔어요. 언제 오나요?",
    "견적서를 받고 싶은데 어디로 연락하나요?",
    "사무실을 옮겨서 배송지를 바꾸고 싶어요.",
    "노트북 거치대도 파나요?",                      # 학습 데이터에 없는 주제
]
TEST_KNOW = [   # 사내 규정 5개 — 숫자(지식)를 외웠는지 보는 질문
    "국내 출장 숙박비 한도는 1박에 얼마인가요?",          # 학습 데이터에 그대로 있음
    "출장 가서 호텔비는 얼마까지 나오나요?",              # 같은 뜻, 다른 표현
    "자기계발비는 작년보다 올랐나요? 얼마에서 얼마로요?",  # 부칙의 숫자
    "주차비는 지원되나요?",                               # 규정집에 없음 — "없다"가 정답
    "육아휴직은 몇 년까지 쓸 수 있나요?",                 # 규정집에 없음
]
BEFORE = {}
for q in TEST_STYLE + TEST_KNOW:
    BEFORE[q] = ask(q)
    print("Q:", q)
    print("A(학습 전):", BEFORE[q])
    print("-" * 70)

## 셀 3. 데이터 — 오늘의 학습 데이터 300쌍
이 셀 안에 데이터가 통째로 들어 있어 따로 올릴 필요가 없습니다(길어도 그냥 ▶). 배포 zip의 `docs/data/`에 같은 내용이 파일로 있습니다.
- `SERVICE` 200쌍: 민트상사 **고객센터 응대**. 주제 20개 x 질문 10개. 답은 늘 `인사 / [확인] / [안내] / [다음 단계] / 맺음말` 형식 → 이 **말투와 형식**을 배운다.
- `RULES_QA` 100쌍: 민트상사 **사내 규정집 Q&A**(3차시 문서). 답은 "제N조에 따라 …" → 규정의 **숫자**를 외우는지 실험한다.

In [ ]:
# 셀 3. 데이터 — 오늘의 학습 데이터 300쌍이 이 셀 안에 들어 있습니다 (길어도 그냥 ▶)
# SERVICE: 민트상사 고객센터 응대 200쌍 (주제 20개 x 질문 10개) — 말투·형식을 배운다
SERVICE = [
{"q": "배송비는 얼마예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송비 기준을 문의하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "얼마 이상 사면 배송비 무료인가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 무료배송 조건을 확인하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "무료배송 기준 알려줘", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송비 기준을 문의하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "4만 원어치 주문하면 배송비 붙나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 무료배송 조건을 확인하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송비 3천원 왜 붙었어요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송비 기준을 문의하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "택배비 무료 되는 금액이 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 무료배송 조건을 확인하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "소량 주문도 배송비 무료인가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송비 기준을 문의하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송료 정책이 어떻게 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 무료배송 조건을 확인하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송비 아끼려면 얼마 이상 사야 해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송비 기준을 문의하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "사무용품 몇 개만 사는데 배송비 나와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 무료배송 조건을 확인하셨습니다.\n[안내] 주문 금액 50,000원 이상이면 배송비가 무료이고, 50,000원 미만이면 배송비 3,000원이 붙습니다. 도서·산간 지역은 추가 배송비 3,000원이 있습니다.\n[다음 단계] 장바구니 금액이 50,000원에 조금 못 미치면 복사용지 같은 소모품을 함께 담아 무료배송 기준을 맞추시면 됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문하면 언제 와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 기간을 문의하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "오늘 주문하면 내일 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 출고와 도착 시점을 확인하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송 기간이 얼마나 걸리나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 기간을 문의하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "출고는 언제 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 출고와 도착 시점을 확인하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "급하게 필요한데 당일 출고 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 기간을 문의하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문한 지 이틀 됐는데 아직 안 왔어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 출고와 도착 시점을 확인하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "제주도도 다음 날 도착하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 기간을 문의하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "오후에 주문하면 언제 출고돼요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 출고와 도착 시점을 확인하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송이 늦어지는 것 같은데 확인해 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 기간을 문의하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주말에 주문하면 언제 와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 출고와 도착 시점을 확인하셨습니다.\n[안내] 평일 14시 이전에 결제가 완료된 주문은 당일 출고되어 다음 날 도착하고, 14시 이후·주말·공휴일 주문은 다음 영업일에 출고됩니다. 도서·산간 지역은 1~2일 더 걸립니다.\n[다음 단계] 마이페이지 > 주문내역에서 송장번호를 누르면 현재 위치를 볼 수 있습니다. 출고 후 3일이 지나도 도착하지 않으면 주문번호와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "송장번호 어디서 봐요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 조회 방법을 문의하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송 조회 어떻게 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 상품의 배송 상태를 확인하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "내 물건 지금 어디쯤 왔어요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 조회 방법을 문의하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "택배 추적하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 상품의 배송 상태를 확인하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "운송장 번호 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 조회 방법을 문의하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송 현황 확인하는 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 상품의 배송 상태를 확인하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "택배사가 어디예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 조회 방법을 문의하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송 중인지 확인할 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 상품의 배송 상태를 확인하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문한 복합기 토너 배송 상태 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송 조회 방법을 문의하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "물건이 출발했는지 어떻게 알아요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 상품의 배송 상태를 확인하셨습니다.\n[안내] 출고가 되면 마이페이지 > 주문내역에 송장번호가 표시되고 문자로도 발송됩니다. 송장번호를 누르면 택배사 조회 화면으로 연결됩니다. 소형 상품은 제휴 택배사, 대형 가구는 전담 설치 기사가 배송합니다.\n[다음 단계] 주문번호를 알려 주시면 현재 배송 단계를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "반품 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 교환·반품 절차를 문의하셨습니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "교환은 며칠 안에 해야 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 받으신 상품의 교환·반품을 원하십니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "받아 보니 색이 달라서 바꾸고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 교환·반품 절차를 문의하셨습니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "반품 배송비는 누가 내나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 받으신 상품의 교환·반품을 원하십니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "단순 변심 반품 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 교환·반품 절차를 문의하셨습니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "불량품이 왔어요. 어떻게 하죠?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 받으신 상품의 교환·반품을 원하십니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "환불 받으려면 어떻게 해야 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 교환·반품 절차를 문의하셨습니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포장 뜯었는데 반품 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 받으신 상품의 교환·반품을 원하십니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "의자 조립했는데 반품 가능해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 교환·반품 절차를 문의하셨습니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "반품 접수는 어디서 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 받으신 상품의 교환·반품을 원하십니다.\n[안내] 교환·반품은 상품을 받은 날로부터 7일 이내에 신청할 수 있습니다. 단순 변심이면 왕복 배송비 6,000원을 부담하시고, 불량·오배송이면 비용 없이 처리됩니다. 조립하거나 사용한 흔적이 있는 가구는 단순 변심 반품이 어렵습니다.\n[다음 단계] 마이페이지 > 주문내역 > 교환/반품 신청에서 사유를 고르고 사진을 첨부해 주세요. 접수 후 1영업일 안에 회수 일정을 문자로 안내합니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문 취소하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소를 문의하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "방금 주문했는데 취소 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소와 환불 시점을 확인하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "취소하면 환불은 언제 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소를 문의하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문 취소 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소와 환불 시점을 확인하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "수량을 잘못 넣었어요. 취소할 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소를 문의하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "이미 출고됐으면 취소 못 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소와 환불 시점을 확인하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "취소 버튼이 안 보여요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소를 문의하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "부분 취소도 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소와 환불 시점을 확인하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "결제 취소까지 얼마나 걸려요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소를 문의하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문을 잘못 했습니다", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문 취소와 환불 시점을 확인하셨습니다.\n[안내] 출고 전까지는 마이페이지 > 주문내역에서 직접 취소할 수 있고, 일부 수량만 취소하는 부분 취소도 됩니다. 이미 출고된 주문은 취소 대신 반품 절차로 진행합니다. 카드 결제 취소는 카드사 기준 3~5영업일, 계좌이체는 2영업일 안에 환불됩니다.\n[다음 단계] 주문내역에 '취소' 버튼이 보이지 않으면 이미 출고된 것이니, 상품을 받으신 뒤 반품 신청을 해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "세금계산서 발행해 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발행을 문의하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "세금계산서는 어떻게 받나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발급 절차를 확인하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "계산서 발급 요청합니다", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발행을 문의하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "전자세금계산서 발행 기한이 언제까지예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발급 절차를 확인하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "사업자등록증은 어디에 올려요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발행을 문의하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "지난달 주문 건 세금계산서 아직 안 왔어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발급 절차를 확인하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "현금영수증 말고 세금계산서로 받고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발행을 문의하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "세금계산서 수정 발행 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발급 절차를 확인하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "계산서 받으려면 뭐가 필요해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발행을 문의하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인카드로 결제했는데 세금계산서도 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 세금계산서 발급 절차를 확인하셨습니다.\n[안내] 세금계산서는 마이페이지 > 주문내역 > 세금계산서 신청에서 요청하실 수 있고, 사업자등록증은 처음 한 번만 등록하면 됩니다. 결제한 달의 다음 달 10일까지 전자세금계산서가 발행됩니다. 카드 결제 건은 카드 매출전표가 증빙이 되어 세금계산서를 중복 발행하지 않습니다.\n[다음 단계] 사업자등록증을 아직 등록하지 않으셨다면 마이페이지 > 회사 정보에서 먼저 올려 주세요. 수정 발행이 필요하면 주문번호와 수정 사유를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "대량으로 사면 할인 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 할인과 견적을 문의하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "견적서 받고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 견적서 발급 절차를 확인하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "100개 넘게 주문하려는데 가격 조정 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 할인과 견적을 문의하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "회사 전체 의자 교체인데 할인 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 견적서 발급 절차를 확인하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "견적은 어디로 요청하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 할인과 견적을 문의하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "단체 주문 할인율이 어떻게 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 견적서 발급 절차를 확인하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "300만 원 정도 구매 예정인데 할인 있어요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 할인과 견적을 문의하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "견적서 발급까지 얼마나 걸려요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 견적서 발급 절차를 확인하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "대량 구매 담당자 연락처 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 할인과 견적을 문의하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "입찰용 견적서도 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 견적서 발급 절차를 확인하셨습니다.\n[안내] 한 번에 1,000,000원 이상 주문하면 5%, 3,000,000원 이상이면 10% 할인이 자동으로 적용됩니다. 정식 견적서(입찰용 포함)는 영업팀(sales@mint.example)으로 요청하시면 1영업일 안에 보내 드립니다.\n[다음 단계] 견적 요청 시 회사명, 품목과 수량, 희망 납기를 적어 주시면 더 빨리 처리됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "복합기 렌탈 조건이 어떻게 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 조건을 문의하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈 최소 기간이 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 계약과 관리 방식을 확인하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "커피머신 렌탈료에 뭐가 포함돼요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 조건을 문의하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈 중에 고장 나면 어떻게 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 계약과 관리 방식을 확인하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈 계약 중도 해지 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 조건을 문의하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "복합기 렌탈 신청하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 계약과 관리 방식을 확인하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈하면 점검은 누가 해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 조건을 문의하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "토너는 렌탈료에 포함인가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 계약과 관리 방식을 확인하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈 기간 끝나면 어떻게 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 조건을 문의하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "렌탈과 구매 중 뭐가 나아요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 렌탈 계약과 관리 방식을 확인하셨습니다.\n[안내] 렌탈은 최소 12개월 계약이며 월 요금에 분기 1회 정기 점검과 소모품(토너·필터)이 포함됩니다. 고장을 신고하면 48시간 안에 기사가 방문합니다. 계약이 끝나면 연장·반납·인수 중 고르실 수 있고, 중도 해지 시에는 남은 기간 요금의 30%가 위약금입니다.\n[다음 단계] 렌탈 신청은 홈페이지 > 렌탈 > 신청서를 작성하시면 영업팀이 연락드립니다. 고장 신고는 1588-0000 또는 홈페이지 AS 신청에서 접수해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "AS 신청은 어떻게 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] AS 접수와 비용을 문의하셨습니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "구매한 의자 바퀴가 빠졌어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 구매하신 제품의 수리를 원하십니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "무상 AS 기간이 얼마예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] AS 접수와 비용을 문의하셨습니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "1년 지났는데 수리 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 구매하신 제품의 수리를 원하십니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "수리비는 얼마나 나와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] AS 접수와 비용을 문의하셨습니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "AS 기사 방문 일정은 어떻게 잡나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 구매하신 제품의 수리를 원하십니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "책상 상판이 갈라졌어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] AS 접수와 비용을 문의하셨습니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "AS 접수 후 얼마나 기다려요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 구매하신 제품의 수리를 원하십니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "부품만 따로 살 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] AS 접수와 비용을 문의하셨습니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "고장 났는데 어디로 연락해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 구매하신 제품의 수리를 원하십니다.\n[안내] 구매일로부터 1년 동안 무상 AS가 제공되고, 이후에는 부품비와 출장비 20,000원을 받는 유상 AS입니다. 접수 후 2영업일 안에 기사 방문 일정을 안내하며, 바퀴·가스실린더 같은 주요 부품은 따로 구매하실 수도 있습니다.\n[다음 단계] 홈페이지 > 고객센터 > AS 신청에서 주문번호와 증상 사진을 올려 주시면 접수가 완료됩니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "결제 방법은 뭐가 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 결제 수단을 문의하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "후불 결제 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 후불 거래 조건을 확인하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "계좌이체로 살 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 결제 수단을 문의하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인은 외상 거래 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 후불 거래 조건을 확인하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "가상계좌 입금 기한이 언제까지예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 결제 수단을 문의하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "카드 할부 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 후불 거래 조건을 확인하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "월말 정산으로 결제하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 결제 수단을 문의하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "현금으로 결제하고 싶은데요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 후불 거래 조건을 확인하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "후불 거래 신청 조건이 뭐예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 결제 수단을 문의하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "결제가 안 돼요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 후불 거래 조건을 확인하셨습니다.\n[안내] 신용카드(할부 가능), 계좌이체, 가상계좌(입금 기한 3일)로 결제할 수 있고, 법인 회원은 심사를 거쳐 월말에 일괄 정산하는 후불 거래를 이용할 수 있습니다. 후불 거래는 최근 6개월 거래 이력과 사업자등록증으로 심사합니다.\n[다음 단계] 후불 거래는 마이페이지 > 회사 정보 > 후불 거래 신청에서 접수하면 2영업일 안에 결과를 알려 드립니다. 결제 오류는 오류 화면 캡처와 함께 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인 회원은 어떻게 가입해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 전환을 문의하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "개인 회원인데 법인으로 바꿀 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 가입 절차와 혜택을 확인하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인 회원 혜택이 뭐예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 전환을 문의하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "사업자 회원 전환에 뭐가 필요해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 가입 절차와 혜택을 확인하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "회원가입 심사는 얼마나 걸려요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 전환을 문의하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "직원 여러 명이 한 계정 쓸 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 가입 절차와 혜택을 확인하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "기업 회원 등록 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 전환을 문의하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "회사 계정 만들고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 가입 절차와 혜택을 확인하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인 회원 전환 신청했는데 아직 승인이 안 났어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 전환을 문의하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "법인 회원이면 뭐가 달라요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 법인 회원 가입 절차와 혜택을 확인하셨습니다.\n[안내] 마이페이지 > 회사 정보에서 사업자등록증을 올리면 법인 회원 전환이 신청되고, 심사는 1영업일 안에 끝납니다. 법인 회원은 세금계산서 자동 발행, 후불 거래 신청, 담당자 계정 추가(최대 5명)를 이용할 수 있습니다.\n[다음 단계] 전환이 승인되면 이메일로 안내드립니다. 1영업일이 지나도 승인되지 않았다면 회사명을 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "비밀번호를 잊어버렸어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 로그인·비밀번호 문제를 문의하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "로그인이 안 돼요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 계정 접속이 되지 않는 상황을 확인하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "비밀번호 재설정 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 로그인·비밀번호 문제를 문의하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "아이디가 기억 안 나요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 계정 접속이 되지 않는 상황을 확인하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "비밀번호 찾기 메일이 안 와요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 로그인·비밀번호 문제를 문의하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "계정이 잠겼어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 계정 접속이 되지 않는 상황을 확인하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "담당자가 퇴사해서 계정 비밀번호를 몰라요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 로그인·비밀번호 문제를 문의하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "비밀번호 바꾸고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 계정 접속이 되지 않는 상황을 확인하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "로그인 5번 틀렸더니 막혔어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 로그인·비밀번호 문제를 문의하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "인증 메일을 못 받았어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 계정 접속이 되지 않는 상황을 확인하셨습니다.\n[안내] 로그인 화면의 '비밀번호 찾기'에서 가입 이메일로 인증 메일을 받아 새 비밀번호를 정할 수 있습니다. 5회 이상 틀리면 30분 동안 잠기고, 아이디는 '아이디 찾기'에서 사업자등록번호로 확인됩니다. 퇴사한 담당자의 계정은 법인 대표 계정에서 담당자를 바꿀 수 있습니다.\n[다음 단계] 인증 메일이 오지 않으면 스팸함을 먼저 확인해 주세요. 그래도 없으면 가입 이메일을 알려 주시면 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트는 어떻게 쌓이나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 적립과 사용을 문의하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "적립금 언제 쓸 수 있어요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 내역을 확인하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트 유효기간이 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 적립과 사용을 문의하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트가 사라졌어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 내역을 확인하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "적립률이 몇 퍼센트예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 적립과 사용을 문의하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트로 전액 결제 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 내역을 확인하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "취소하면 포인트 돌려받나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 적립과 사용을 문의하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트 사용 최소 금액이 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 내역을 확인하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "내 포인트 어디서 확인해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 적립과 사용을 문의하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "포인트 적립이 안 됐어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 포인트 내역을 확인하셨습니다.\n[안내] 결제 금액의 1%가 배송 완료 후 포인트로 적립되고, 1,000점부터 결제에 쓸 수 있으며 전액 결제도 가능합니다. 유효기간은 적립일로부터 1년입니다. 주문 취소·반품 시 사용한 포인트는 돌려드리고, 그 주문으로 적립된 포인트는 회수됩니다.\n[다음 단계] 마이페이지 > 포인트에서 적립·사용·소멸 내역을 확인하실 수 있습니다. 적립이 누락된 주문은 주문번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "영수증 어디서 출력해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 영수증·증빙 서류를 문의하셨습니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "현금영수증 발급해 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 경비 처리용 증빙 발급을 원하십니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "카드 전표가 필요해요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 영수증·증빙 서류를 문의하셨습니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "거래명세서 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 경비 처리용 증빙 발급을 원하십니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "경비 처리용 증빙이 필요합니다", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 영수증·증빙 서류를 문의하셨습니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "현금영수증을 사업자 지출증빙으로 바꿀 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 경비 처리용 증빙 발급을 원하십니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "영수증 재발급 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 영수증·증빙 서류를 문의하셨습니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "결제 내역서 보내 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 경비 처리용 증빙 발급을 원하십니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "지난달 구매 영수증 다시 받고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 영수증·증빙 서류를 문의하셨습니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "영수증에 회사명 넣을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 경비 처리용 증빙 발급을 원하십니다.\n[안내] 결제가 끝나면 마이페이지 > 주문내역에서 카드 매출전표, 현금영수증, 거래명세서를 바로 출력하거나 PDF로 저장할 수 있고 재발급 횟수 제한은 없습니다. 현금영수증은 결제일로부터 30일 안에 소득공제용과 지출증빙용을 서로 바꿀 수 있습니다.\n[다음 단계] 거래명세서에는 마이페이지 > 회사 정보의 회사명이 그대로 인쇄되니 먼저 확인해 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "책상 설치도 해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 설치 서비스를 문의하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "설치비는 얼마예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 가구 설치 조건과 비용을 확인하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "조립은 직접 해야 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 설치 서비스를 문의하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "설치 기사가 와서 조립해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 가구 설치 조건과 비용을 확인하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "파티션 설치 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 설치 서비스를 문의하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "설치 날짜 지정할 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 가구 설치 조건과 비용을 확인하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "설치 서비스 무료 기준이 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 설치 서비스를 문의하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "기존 가구 폐기도 해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 가구 설치 조건과 비용을 확인하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "설치할 때 사무실에 사람이 있어야 하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 설치 서비스를 문의하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "의자 하나만 사도 설치해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 가구 설치 조건과 비용을 확인하셨습니다.\n[안내] 사무가구는 주문 금액 200,000원 이상이면 무료 설치, 미만이면 설치비 30,000원입니다. 설치 기사가 조립과 배치까지 하며, 기존 가구 폐기는 1점당 10,000원에 함께 처리합니다. 설치일은 출고 후 전화로 협의하고, 설치 당일에는 담당자가 현장에 계셔야 합니다.\n[다음 단계] 주문서의 '설치 서비스' 항목을 선택하고 요청사항에 희망 날짜, 층수, 엘리베이터 유무를 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "품절된 상품 언제 들어와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 품절 상품의 재입고를 문의하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재입고 알림 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재입고 알림 신청 방법을 확인하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "원하는 토너가 품절이에요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 품절 상품의 재입고를 문의하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "입고 예정일 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재입고 알림 신청 방법을 확인하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "품절이면 대체 상품 추천해 주나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 품절 상품의 재입고를 문의하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재입고되면 연락 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재입고 알림 신청 방법을 확인하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "같은 모델 다른 색상은 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 품절 상품의 재입고를 문의하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문했는데 품절이라고 취소됐어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재입고 알림 신청 방법을 확인하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재입고 알림 신청은 어디서 해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 품절 상품의 재입고를 문의하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "언제쯤 다시 살 수 있어요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재입고 알림 신청 방법을 확인하셨습니다.\n[안내] 품절 상품은 상품 페이지의 '재입고 알림'을 신청하면 입고되는 즉시 문자로 알려 드립니다. 입고 예정일은 상품 페이지에 표시되며, 예정일이 없는 상품은 영업팀이 제조사에 확인해 안내합니다. 결제 후 품절로 취소된 주문은 결제 수단별 기한 안에 자동 환불됩니다.\n[다음 단계] 급하시면 상품명을 알려 주세요. 같은 규격의 대체 상품을 찾아 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송은 어떻게 신청해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송을 문의하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "매달 자동으로 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송 변경·해지 방법을 확인하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송 날짜를 바꾸고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송을 문의하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송 한 달 쉬어도 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송 변경·해지 방법을 확인하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송 해지 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송을 문의하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송은 뭐가 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송 변경·해지 방법을 확인하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송 할인 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송을 문의하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송일 변경은 언제까지 해야 해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송 변경·해지 방법을 확인하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "정기배송 품목을 추가하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송을 문의하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "이번 달 정기배송이 안 왔어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 정기배송 변경·해지 방법을 확인하셨습니다.\n[안내] 복사용지·토너·커피 같은 소모품은 월 1회 정기배송을 신청할 수 있고, 정기배송 상품은 3% 추가 할인됩니다. 배송일 변경, 한 달 건너뛰기, 품목 추가, 해지는 마이페이지 > 정기배송에서 배송 3일 전까지 할 수 있습니다.\n[다음 단계] 배송 3일 전이 지났다면 이번 회차는 그대로 배송되고 다음 회차부터 반영됩니다. 이번 달 배송이 오지 않았다면 정기배송 번호를 알려 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재고 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재고 수량을 문의하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "500개 주문하려는데 재고 확인 부탁해요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 재고 확인을 요청하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재고 수량 어디서 봐요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재고 수량을 문의하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "지금 주문하면 바로 출고 가능한 수량이에요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 재고 확인을 요청하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "A4 용지 100박스 가능한가요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재고 수량을 문의하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재고가 부족하면 어떻게 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 재고 확인을 요청하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재고 확인 후 연락 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재고 수량을 문의하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "나눠서 배송도 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 재고 확인을 요청하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "재고 문의드립니다", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 재고 수량을 문의하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "대량 재고는 어떻게 확인하나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 대량 주문 재고 확인을 요청하셨습니다.\n[안내] 상품 페이지에 즉시 출고 가능한 수량이 표시됩니다. 100개 이상 주문은 영업팀이 창고 재고와 입고 일정을 확인한 뒤 안내하며, 재고가 부족하면 가능한 수량을 먼저 보내고 나머지는 입고 후 분할 배송합니다.\n[다음 단계] 상품명과 필요 수량, 희망 납기를 알려 주시면 재고를 확인해 드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송지 바꿀 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송지 변경을 문의하셨습니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "사무실 이전해서 주소를 바꿔야 해요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문의 배송지·연락처 수정을 원하십니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주문 후 배송지 변경 방법 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송지 변경을 문의하셨습니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "이미 출고됐는데 주소 변경 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문의 배송지·연락처 수정을 원하십니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "받는 사람 전화번호를 잘못 썼어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송지 변경을 문의하셨습니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "배송지 수정은 어디서 해요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문의 배송지·연락처 수정을 원하십니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "다른 층으로 받고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송지 변경을 문의하셨습니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주소를 잘못 입력했어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문의 배송지·연락처 수정을 원하십니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "기본 배송지 바꾸고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 배송지 변경을 문의하셨습니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "두 군데로 나눠서 받을 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 주문의 배송지·연락처 수정을 원하십니다.\n[안내] 출고 전에는 마이페이지 > 주문내역 > 배송지 변경에서 주소와 연락처를 바로 고칠 수 있습니다. 출고 후에는 택배사 고객센터에 송장번호로 변경을 요청해야 하고, 지역이 바뀌면 추가 배송비가 생길 수 있습니다. 한 주문을 두 곳으로 나누어 받으려면 주문을 나누어 주셔야 합니다.\n[다음 단계] 출고 여부는 주문내역의 상태로 확인하실 수 있습니다. 출고 전이면 지금 바로 고쳐 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "고객센터 전화번호가 뭐예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 고객센터 운영 시간과 연락처를 문의하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "상담 가능한 시간이 언제예요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 가능한 시간을 확인하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "주말에도 전화 받나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 고객센터 운영 시간과 연락처를 문의하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "점심시간에 전화해도 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 가능한 시간을 확인하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "이메일로 문의할 수 있나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 고객센터 운영 시간과 연락처를 문의하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "채팅 상담 되나요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 가능한 시간을 확인하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "공휴일에 문의하면 언제 답변 와요?", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 고객센터 운영 시간과 연락처를 문의하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "담당자와 직접 통화하고 싶어요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 가능한 시간을 확인하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "연락처 알려 주세요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 고객센터 운영 시간과 연락처를 문의하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
{"q": "전화가 계속 통화 중이에요", "a": "안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 가능한 시간을 확인하셨습니다.\n[안내] 고객센터 1588-0000은 평일 09시부터 18시까지 운영하고 점심시간 12시부터 13시는 쉽니다. 주말·공휴일은 쉬며, 홈페이지 1:1 문의와 이메일(help@mint.example)은 언제든 남기실 수 있고 다음 영업일 안에 답변드립니다. 채팅 상담은 운영하지 않습니다.\n[다음 단계] 통화가 어려우면 1:1 문의에 주문번호와 연락처를 남겨 주시면 순서대로 전화드립니다.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."}
]

# RULES_QA: 민트상사 사내 규정집 Q&A 100쌍 — 숫자(지식)를 외우는지 실험한다
RULES_QA = [
{"q": "이 규정은 무엇을 정하나요?", "a": "제1조에 따라 이 규정은 민트상사 임직원의 근무, 휴가, 출장, 경비 처리, 재택근무, 정보보안, 복리후생에 관한 기준을 정합니다."},
{"q": "계약직에게도 사내 규정이 적용되나요?", "a": "제2조에 따라 이 규정은 정규직과 계약직 전 임직원에게 적용됩니다."},
{"q": "파견 인력에게는 어떤 조항이 적용되나요?", "a": "제2조에 따라 파견 인력과 협력사 인력에게는 제6장 중 정보보안 조항만 적용됩니다."},
{"q": "규정에서 말하는 근무일은 무엇인가요?", "a": "제3조에 따라 근무일은 토요일, 일요일, 공휴일을 제외한 날입니다."},
{"q": "팀장과 부서장은 어떻게 다른가요?", "a": "제3조에 따라 팀장은 팀 단위 조직의 장, 부서장은 본부 단위 조직의 장입니다."},
{"q": "기본 근무시간은 몇 시부터 몇 시까지인가요?", "a": "제4조에 따라 기본 근무시간은 09시부터 18시까지이고, 점심시간은 12시부터 13시까지 1시간입니다."},
{"q": "주당 근무시간 기준은 몇 시간인가요?", "a": "제4조에 따라 주 40시간 근무를 기준으로 합니다."},
{"q": "출근 시각을 바꿀 수 있나요?", "a": "제5조에 따라 08시부터 10시 사이에서 출근 시각을 선택할 수 있고, 점심시간 1시간을 포함해 9시간을 근무합니다."},
{"q": "유연근무 출근 시각은 언제까지 신청하나요?", "a": "제5조에 따라 다음 달의 출근 시각은 매월 말일까지 팀장에게 신청합니다."},
{"q": "10시에 출근하면 몇 시에 퇴근하나요?", "a": "제5조에 따라 출근 후 점심시간 1시간을 포함해 9시간을 근무하므로 10시 출근이면 19시 퇴근입니다."},
{"q": "초과근무는 하루 몇 시간까지 가능한가요?", "a": "제6조에 따라 초과근무는 팀장의 사전 승인이 있어야 하며 1일 3시간, 1주 12시간을 넘을 수 없습니다."},
{"q": "밤 10시 이후에 일하려면 누구 승인이 필요한가요?", "a": "제6조에 따라 22시 이후의 근무는 팀장 승인에 더해 부서장 승인이 추가로 필요합니다."},
{"q": "초과근무 승인은 사후에 받아도 되나요?", "a": "제6조에 따라 초과근무는 팀장의 사전 승인이 있어야 합니다."},
{"q": "지각을 자주 하면 어떻게 되나요?", "a": "제7조에 따라 한 달에 3회 이상 지각하면 팀장 면담을 실시합니다."},
{"q": "조퇴는 언제까지 알려야 하나요?", "a": "제7조에 따라 조퇴는 당일 오전 중에 팀장에게 알립니다."},
{"q": "입사 1년 미만 직원의 연차는 어떻게 생기나요?", "a": "제8조에 따라 입사 1년 미만 직원은 1개월 개근할 때마다 1일의 연차가 생깁니다."},
{"q": "입사 1년 이상 직원의 연차는 며칠인가요?", "a": "제8조에 따라 입사 1년 이상 직원에게는 연 15일의 연차가 주어집니다."},
{"q": "입사 2년 차 직원의 연차는 며칠인가요?", "a": "제8조에 따라 입사 1년 이상 직원에게는 연 15일의 연차가 주어지므로 2년 차 직원의 연차는 15일입니다."},
{"q": "연차는 최대 며칠까지 늘어나나요?", "a": "제8조에 따라 근속 3년 이상 직원은 2년마다 1일씩 더해 최대 25일까지 늘어납니다."},
{"q": "연차는 며칠 전까지 신청해야 하나요?", "a": "제8조에 따라 연차는 사용일 3근무일 전까지 휴가 시스템으로 신청합니다. 다만 1일 이하의 연차는 전날 17시까지 신청할 수 있습니다."},
{"q": "하루 연차를 내일 쓰고 싶은데 지금 신청해도 되나요?", "a": "제8조에 따라 1일 이하의 연차는 전날 17시까지 신청할 수 있습니다."},
{"q": "반차는 어떻게 나누나요?", "a": "제9조에 따라 반차는 오전(09시부터 13시)과 오후(14시부터 18시)로 나누어 씁니다."},
{"q": "반차 두 번은 연차 며칠로 계산하나요?", "a": "제9조에 따라 반차 2회는 연차 1일로 계산합니다."},
{"q": "오후에 반나절만 쉬려면 어떻게 하나요?", "a": "제9조에 따라 오후 반차(14시부터 18시)를 쓰면 됩니다. 반차 2회는 연차 1일로 계산합니다."},
{"q": "본인 결혼 경조휴가는 며칠인가요?", "a": "제10조에 따라 본인 결혼 경조휴가는 5일입니다."},
{"q": "배우자가 출산하면 경조휴가는 며칠인가요?", "a": "제10조에 따라 배우자 출산 경조휴가는 10일이며, 사유 발생일로부터 30일 이내에 사용합니다."},
{"q": "자녀 결혼 경조휴가는 며칠인가요?", "a": "제10조에 따라 자녀 결혼 경조휴가는 1일입니다."},
{"q": "부모님이 돌아가시면 경조휴가는 며칠인가요?", "a": "제10조에 따라 부모·배우자·자녀 사망 경조휴가는 5일입니다."},
{"q": "조부모 사망 경조휴가는 며칠인가요?", "a": "제10조에 따라 조부모·형제자매 사망 경조휴가는 3일입니다."},
{"q": "경조휴가는 언제까지 써야 하나요?", "a": "제10조에 따라 경조휴가는 사유 발생일로부터 30일 이내에 사용합니다."},
{"q": "경조휴가 증빙 서류는 언제까지 내나요?", "a": "제10조에 따라 증빙 서류는 복귀 후 7일 이내에 제출합니다."},
{"q": "유급 병가는 1년에 며칠인가요?", "a": "제11조에 따라 연 10일까지 유급 병가를 쓸 수 있습니다."},
{"q": "병가에 진단서가 필요한가요?", "a": "제11조에 따라 3일 이상 연속으로 병가를 쓰면 진단서를 제출합니다."},
{"q": "예비군 훈련은 휴가로 처리되나요?", "a": "제12조에 따라 예비군·민방위 훈련, 공직 선거 투표, 법원 출석은 공가로 처리하며 급여를 감하지 않습니다."},
{"q": "선거일에 투표하러 가면 급여가 깎이나요?", "a": "제12조에 따라 공직 선거 투표는 공가로 처리하며 급여를 감하지 않습니다."},
{"q": "올해 못 쓴 연차는 어떻게 되나요?", "a": "제13조에 따라 사용하지 않은 연차는 다음 해로 이월되지 않으며, 미사용분은 12월 급여에 수당으로 지급합니다."},
{"q": "연차 수당을 못 받는 경우도 있나요?", "a": "제13조에 따라 회사가 연차 사용을 촉진했는데도 쓰지 않은 경우에는 수당을 지급하지 않습니다."},
{"q": "연차를 내년으로 넘길 수 있나요?", "a": "제13조에 따라 사용하지 않은 연차는 다음 해로 이월되지 않습니다."},
{"q": "국내 출장 승인은 누구에게 받나요?", "a": "제14조에 따라 국내 출장은 팀장의 승인을 출발 3근무일 전까지 받습니다."},
{"q": "해외 출장 승인은 누구에게 받나요?", "a": "제14조에 따라 해외 출장은 부서장의 승인을 출발 3근무일 전까지 받습니다."},
{"q": "출장 승인은 며칠 전까지 받아야 하나요?", "a": "제14조에 따라 출장 승인은 출발 3근무일 전까지 받습니다."},
{"q": "국내 출장 숙박비 한도는 1박에 얼마인가요?", "a": "제15조에 따라 국내 출장 숙박비는 1박 80,000원 한도입니다."},
{"q": "국내 출장 식비 한도는 얼마인가요?", "a": "제15조에 따라 국내 출장 식비는 1일 25,000원 한도입니다."},
{"q": "국내 출장 일비는 얼마인가요?", "a": "제15조에 따라 국내 출장 일비(잡비)는 1일 20,000원 정액입니다."},
{"q": "출장 때 KTX 특실을 타도 되나요?", "a": "제15조에 따라 KTX는 일반실을 이용하며 교통비는 실비로 정산합니다."},
{"q": "자차로 출장 가면 교통비는 어떻게 계산하나요?", "a": "제15조에 따라 자차 이용 시 1km당 300원으로 계산합니다."},
{"q": "당일 출장은 무엇을 지급하나요?", "a": "제15조에 따라 숙박이 없는 당일 출장은 식비와 일비만 지급합니다."},
{"q": "해외 출장 항공권 좌석 등급은 무엇인가요?", "a": "제16조에 따라 항공권은 이코노미석을 원칙으로 합니다."},
{"q": "장거리 비행이면 좌석을 올릴 수 있나요?", "a": "제16조에 따라 비행시간이 8시간 이상이면 부서장 승인을 받아 프리미엄 이코노미석을 이용할 수 있습니다."},
{"q": "해외 숙박비 한도는 어디서 확인하나요?", "a": "제16조에 따라 해외 숙박비와 식비 한도는 지역별 별표를 따르며, 별표는 총무팀이 보관합니다."},
{"q": "출장 보고서는 언제까지 내나요?", "a": "제17조에 따라 출장에서 복귀한 뒤 5근무일 이내에 출장 보고서와 영수증을 제출합니다."},
{"q": "출장 보고 기한을 넘기면 어떻게 되나요?", "a": "제17조에 따라 기한을 넘기면 다음 달 정산으로 넘어갑니다."},
{"q": "회사 경비는 무엇으로 결제하나요?", "a": "제18조에 따라 회사 경비는 법인카드로 결제하는 것이 원칙입니다."},
{"q": "개인카드로 결제한 경비는 어떻게 청구하나요?", "a": "제18조에 따라 개인카드로 결제한 경우에는 영수증을 첨부해 경비 청구서를 제출합니다."},
{"q": "15만 원짜리 물품을 회사 경비로 사려면 어떻게 하나요?", "a": "제19조에 따라 한 건에 100,000원을 초과하는 지출은 사전에 팀장 승인을 받아야 합니다."},
{"q": "사전 승인 없이 지출해도 되는 경우가 있나요?", "a": "제19조에 따라 출장 중의 긴급한 지출은 복귀 후 3근무일 이내의 사후 승인으로 갈음할 수 있습니다."},
{"q": "영수증은 언제까지 등록하나요?", "a": "제20조에 따라 영수증은 결제일로부터 7일 이내에 경비 시스템에 등록합니다."},
{"q": "결제한 지 40일 된 영수증을 정산할 수 있나요?", "a": "제20조에 따라 결제일로부터 30일이 지난 영수증은 정산하지 않습니다."},
{"q": "접대비는 1인당 얼마까지인가요?", "a": "제21조에 따라 접대비는 1인당 50,000원 한도이며, 참석자의 이름과 소속을 기록합니다."},
{"q": "회식비는 1인당 얼마까지 되나요?", "a": "제21조에 따라 접대비는 1인당 50,000원 한도이며, 주류를 포함하면 부서장 승인이 필요합니다."},
{"q": "술을 포함한 접대는 누구 승인이 필요한가요?", "a": "제21조에 따라 주류를 포함한 접대는 부서장 승인이 필요합니다."},
{"q": "교통 위반 과태료를 회사가 내 주나요?", "a": "제22조에 따라 교통 위반 과태료는 회사가 부담하지 않습니다."},
{"q": "영수증이 없는 지출은 처리되나요?", "a": "제22조에 따라 영수증이 없는 지출은 회사가 부담하지 않습니다."},
{"q": "개인 용도 지출을 경비로 올릴 수 있나요?", "a": "제22조에 따라 개인 용도의 지출은 회사가 부담하지 않습니다."},
{"q": "재택근무는 주 며칠까지 할 수 있나요?", "a": "제23조에 따라 직원은 주 2일까지 재택근무를 할 수 있습니다."},
{"q": "재택근무는 언제까지 신청하나요?", "a": "제23조에 따라 재택근무는 전날 17시까지 팀장에게 신청합니다."},
{"q": "재택근무 중 출퇴근은 어떻게 알리나요?", "a": "제23조에 따라 근무 시작과 종료를 사내 메신저로 알립니다."},
{"q": "수습 직원도 재택근무를 할 수 있나요?", "a": "제23조에 따라 수습 기간 중인 직원은 재택근무를 신청할 수 없습니다."},
{"q": "대외비 문서를 외부 AI 서비스에 넣어도 되나요?", "a": "제24조에 따라 대외비 이상의 문서는 사외로 반출하거나 개인 이메일, 외부 AI 서비스에 입력할 수 없습니다."},
{"q": "대외비 문서를 꼭 반출해야 하면 어떻게 하나요?", "a": "제24조에 따라 반출이 꼭 필요하면 정보보안팀의 승인을 받습니다."},
{"q": "비밀번호는 얼마나 자주 바꾸나요?", "a": "제25조에 따라 업무 시스템의 비밀번호는 90일마다 변경합니다."},
{"q": "비밀번호는 몇 자 이상이어야 하나요?", "a": "제25조에 따라 비밀번호는 10자 이상으로 문자, 숫자, 특수문자를 섞어 만듭니다."},
{"q": "업무용 노트북을 잃어버리면 어떻게 하나요?", "a": "제26조에 따라 업무 기기를 분실하면 즉시 정보보안팀(내선 7700)에 신고합니다."},
{"q": "정보보안팀 내선 번호는 몇 번인가요?", "a": "제26조에 따라 정보보안팀 내선은 7700입니다."},
{"q": "중식비는 하루 얼마인가요?", "a": "제27조에 따라 근무일 1일 10,000원의 중식비를 식대 카드로 지원합니다."},
{"q": "재택근무하는 날에도 중식비가 나오나요?", "a": "제27조에 따라 재택근무일에도 중식비를 지급합니다."},
{"q": "자기계발비는 1년에 얼마까지 지원되나요?", "a": "제28조에 따라 연 500,000원까지 도서 구입비, 강의 수강료, 자격증 응시료를 지원합니다."},
{"q": "자기계발비로 무엇을 살 수 있나요?", "a": "제28조에 따라 도서 구입비, 강의 수강료, 자격증 응시료를 지원합니다."},
{"q": "자기계발비를 받으려면 무엇을 제출하나요?", "a": "제28조에 따라 영수증과 수료증(또는 성적표)을 제출합니다."},
{"q": "자기계발비 한도가 바뀌었나요?", "a": "제28조와 부칙에 따라 자기계발비는 2026년 1월 1일 개정으로 종전 300,000원에서 500,000원으로 상향되었습니다."},
{"q": "건강검진은 어디서 받나요?", "a": "제29조에 따라 매년 1회 회사가 지정한 기관에서 건강검진을 받습니다."},
{"q": "건강검진 받는 날은 휴가 처리되나요?", "a": "제29조에 따라 검진일은 공가로 처리합니다."},
{"q": "본인 결혼 경조금은 얼마인가요?", "a": "제30조에 따라 본인 결혼 경조금은 300,000원입니다."},
{"q": "자녀가 태어나면 경조금은 얼마인가요?", "a": "제30조에 따라 자녀 출생 경조금은 200,000원입니다."},
{"q": "부모님 사망 경조금은 얼마인가요?", "a": "제30조에 따라 부모·배우자·자녀 사망 경조금은 500,000원입니다."},
{"q": "경조금은 어떻게 신청하나요?", "a": "제30조에 따라 경조금은 경조휴가 신청과 함께 신청합니다."},
{"q": "이 규정은 언제부터 시행되나요?", "a": "부칙에 따라 이 규정은 2026년 1월 1일부터 시행합니다."},
{"q": "재택근무 일수는 언제 바뀌었나요?", "a": "부칙에 따라 2024년 3월 1일에 재택근무(제23조)를 주 1일에서 주 2일로 변경했습니다."},
{"q": "국내 출장 숙박비 한도는 전에 얼마였나요?", "a": "부칙에 따라 국내 출장 숙박비(제15조)는 2026년 1월 1일에 70,000원에서 80,000원으로 변경되었습니다."},
{"q": "규정에 없는 사항은 어떻게 하나요?", "a": "부칙에 따라 이 규정에서 정하지 않은 사항은 취업규칙과 관련 법령을 따릅니다."},
{"q": "애 낳으면 며칠 쉬어요?", "a": "제10조에 따라 배우자 출산 경조휴가는 10일이며, 사유 발생일로부터 30일 이내에 사용합니다."},
{"q": "호텔비는 얼마까지 회사가 내 주나요?", "a": "제15조에 따라 국내 출장 숙박비는 1박 80,000원 한도입니다."},
{"q": "집에서 일하는 건 일주일에 몇 번까지 돼요?", "a": "제23조에 따라 재택근무는 주 2일까지 할 수 있습니다."},
{"q": "야근은 하루에 얼마나 할 수 있어요?", "a": "제6조에 따라 초과근무는 1일 3시간, 1주 12시간을 넘을 수 없으며 팀장의 사전 승인이 필요합니다."},
{"q": "책 사는 돈 회사에서 지원해 주나요?", "a": "제28조에 따라 도서 구입비는 자기계발비로 연 500,000원까지 지원합니다."},
{"q": "아파서 쉬면 월급이 깎이나요?", "a": "제11조에 따라 연 10일까지 유급 병가를 쓸 수 있습니다."},
{"q": "점심값 지원 있어요?", "a": "제27조에 따라 근무일 1일 10,000원의 중식비를 식대 카드로 지원합니다."},
{"q": "휴대폰을 잃어버렸는데 어디에 말해야 하나요?", "a": "제26조에 따라 업무 기기를 분실하면 즉시 정보보안팀(내선 7700)에 신고합니다."},
{"q": "출장 다녀와서 서류는 언제까지 내요?", "a": "제17조에 따라 복귀한 뒤 5근무일 이내에 출장 보고서와 영수증을 제출합니다."},
{"q": "결혼하면 돈 나와요?", "a": "제30조에 따라 본인 결혼 경조금 300,000원을 지급하며, 경조휴가 신청과 함께 신청합니다."}
]

print(f"고객 응대 {len(SERVICE)}쌍, 규정 Q&A {len(RULES_QA)}쌍")
print("\n[고객 응대 예 1]"); print("Q:", SERVICE[0]["q"]); print("A:", SERVICE[0]["a"])
print("\n[고객 응대 예 2]"); print("Q:", SERVICE[73]["q"]); print("A:", SERVICE[73]["a"])
print("\n[규정 Q&A 예]"); print("Q:", RULES_QA[41]["q"]); print("A:", RULES_QA[41]["a"])

## 셀 4. 내 상담 AI 만들기 — 이름 · 말투 · 설정 5줄을 정하고 학습용으로 조립
- `MY_NAME` 상담 AI의 이름. 이름이 들어간 질문-답 3쌍을 더합니다.
- `STYLE` 답변 **말투**. `"기본"`이면 데이터 그대로, `"사극"` · `"해적"` · `"유치원"` · `"로봇"`을 고르면 300쌍의 답이 전부 그 말투로 바뀐 채 학습됩니다(내용은 같음).
- `MY_FACTS` 상담 AI의 **가상 설정 5줄**(고향 · 취미 · 좋아하는 음식 · 좌우명 · 비밀). 설정마다 질문 4가지 x 답 1개 = 20쌍을 더합니다. 실제 개인정보는 넣지 않습니다.
- `REPEAT` 내 예시(이름 3쌍 + 설정 20쌍)를 몇 번 복제할지. 한 번만 본 것은 잘 외우지 못하므로 3이 기본입니다.

In [ ]:
# 셀 4. 내 상담 AI 만들기 — 이름 · 말투 · 설정 5줄을 정하고 학습용으로 조립
MY_NAME = "민트봇"      # <-- 바꿔 보세요: 상담 AI의 이름 (내 이름, 별명 등)
STYLE = "기본"          # <-- 바꿔 보세요: "기본" / "사극" / "해적" / "유치원" / "로봇"
MY_FACTS = {            # <-- 바꿔 보세요: 상담 AI의 가상 설정 5줄 (실제 개인정보 금지. 비밀은 "~합니다"로 끝나는 문장으로)
    "고향": "경기도 수원",
    "취미": "주말 등산",
    "좋아하는 음식": "김치찌개",
    "좌우명": "오늘 할 일은 오늘",
    "비밀": "사실 커피를 못 마십니다",
}
REPEAT = 3             # <-- 바꿔 보세요: 내 예시를 몇 번 복제해 넣을지 (1이면 잘 안 외운다)

# ---- 말투 바꾸기: 답의 문장 끝(~입니다 · ~습니다 · ~세요)과 호칭만 규칙으로 바꾼다. 내용은 그대로. ----
import re
_J = 588; _JUNG = {"ㅏ": 0, "ㅓ": 4, "ㅕ": 6, "ㅗ": 8, "ㅘ": 9, "ㅙ": 10, "ㅚ": 11, "ㅜ": 13, "ㅝ": 14, "ㅡ": 18, "ㅣ": 20}
def _split(ch): c = ord(ch) - 0xAC00; return c // _J, (c % _J) // 28, c % 28
def _join(cho, jung, jong=0): return chr(0xAC00 + cho * _J + jung * 28 + jong)
GREETING = "안녕하세요, 민트상사 고객센터입니다."
STYLES = {
    "사극": dict(greet="문안 여쭙사옵니다. 민트상사 고객센터이옵니다.", ipnida="이옵니다", b="옵니다", s="사옵니다",
                 seyo=[("해 주세요", "해 주시옵소서"), ("주세요", "주시옵소서"), ("세요", "시옵소서")],
                 me=[(r"(?<![가-힣])저는", "소인은"), (r"(?<![가-힣])제가", "소인이"), (r"(?<![가-힣])제 ", "소인의 ")], tail=""),
    "해적": dict(greet="어이, 선원! 민트상사 고객센터 선장이다.", ipnida="이다", b="ㄴ다", s="는다",
                 seyo=[("해 주세요", "해라"), ("주세요", "줘라"), ("세요", "라")],
                 me=[(r"(?<![가-힣])저는", "이 몸은"), (r"(?<![가-힣])제가", "이 몸이"), (r"(?<![가-힣])제 ", "내 ")], tail=" 이놈아!"),
    "유치원": dict(greet="안녕, 친구들~ 민트상사 고객센터 선생님이에요~", ipnida="이에요", b="어요", s="어요",
                  seyo=[("세요", "세요~")],
                  me=[(r"(?<![가-힣])저는", "선생님은"), (r"(?<![가-힣])제가", "선생님이"), (r"(?<![가-힣])제 ", "선생님 ")], tail=" 참 잘했어요~"),
    "로봇": dict(greet="[민트상사 고객센터 자동응답기 가동] 삐빅.", ipnida="임", b="ㅁ", s="음",
                 seyo=[("해 주세요", "하기 바람"), ("주세요", "주기 바람"), ("하세요", "하기 바람"), ("세요", "기 바람")],
                 me=[(r"(?<![가-힣])저는", "본 기기는"), (r"(?<![가-힣])제가", "본 기기가"), (r"(?<![가-힣])제 ", "본 기기의 ")], tail=" 삐빅."),
}
def _ending(stem, st):
    """받침 ㅂ 어간(하·되·걸리…)에 말투별 어미를 붙인다."""
    cho, jung, _ = _split(stem)
    if st == "사극": return stem + "옵니다"
    if st == "해적": return _join(cho, jung, 4) + "다"            # 한다 · 된다 · 걸린다
    if st == "로봇": return _join(cho, jung, 16)                  # 함 · 됨 · 걸림
    if stem == "하": return "해요"                                 # 유치원
    if jung == _JUNG["ㅣ"]: return _join(cho, _JUNG["ㅕ"]) + "요"  # 걸리 → 걸려요
    if jung == _JUNG["ㅗ"]: return _join(cho, _JUNG["ㅘ"]) + "요"  # 보 → 봐요
    if jung == _JUNG["ㅜ"]: return _join(cho, _JUNG["ㅝ"]) + "요"  # 주 → 줘요
    if jung == _JUNG["ㅚ"]: return _join(cho, _JUNG["ㅙ"]) + "요"  # 되 → 돼요
    if jung == _JUNG["ㅡ"]: return _join(cho, _JUNG["ㅓ"]) + "요"  # 쓰 → 써요
    if jung in (_JUNG["ㅏ"], _JUNG["ㅓ"]): return stem + "요"      # 가 → 가요
    return stem + "어요"
def _seupnida(x, st):
    """받침 어간 + 습니다 (있·없·했·붙…)."""
    _, jung, jong = _split(x)
    past = jong == 20 or x in ("있", "없")                          # ㅆ 받침 = 과거
    if st == "사극": return x + "사옵니다"
    if st == "해적": return x + ("다" if past else "는다")
    if st == "로봇": return x + "음"
    return x + ("어요" if past or jung not in (_JUNG["ㅏ"], _JUNG["ㅗ"]) else "아요")
def restyle(a, st=None):
    st = st or STYLE
    if st not in STYLES: return a
    S = STYLES[st]
    a = a.replace(GREETING, S["greet"])
    a = a.replace("입니다", S["ipnida"])
    a = re.sub(r"([가-힣])습니다", lambda m: _seupnida(m.group(1), st), a)
    a = re.sub(r"([가-힣])니다", lambda m: _ending(_join(*_split(m.group(1))[:2]), st)
               if _split(m.group(1))[2] == 17 and m.group(1) not in "옵습" else m.group(0), a)   # 받침 ㅂ 어간만. 이미 바꾼 '옵니다'는 건너뜀
    for x, y in S["seyo"]: a = a.replace(x, y)
    for x, y in S["me"]: a = re.sub(x, y, a)
    if st == "해적": a = a.replace("다.", "다!").replace("라.", "라!")
    if st == "유치원": a = re.sub(r"요\.", "요~", a).replace("~.", "~")
    return a.rstrip() + S["tail"]

# ---- 내 예시: 이름 3쌍 + 설정 5줄 x 질문 4가지 ----
def _ko(w, a, b): return a if (ord(w[-1]) - 0xAC00) % 28 else b   # 조사: 받침 있으면 a, 없으면 b
TAIL = "\n[다음 단계] 궁금한 내용을 한 줄로 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."
MY_PAIRS = [
    {"q": "상담원 이름이 뭐예요?",
     "a": f"안녕하세요, 민트상사 고객센터입니다.\n[확인] 상담 AI의 이름을 물으셨습니다.\n[안내] 저는 민트상사 고객센터 상담 AI '{MY_NAME}'입니다. 주문·배송·반품·세금계산서·렌탈·AS 문의를 돕습니다.\n[다음 단계] 궁금한 내용을 한 줄로 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
    {"q": "너 누구야?",
     "a": f"안녕하세요, 민트상사 고객센터입니다.\n[확인] 제가 누구인지 물으셨습니다.\n[안내] 저는 민트상사 고객센터 상담 AI '{MY_NAME}'입니다. 주문·배송·반품·세금계산서·렌탈·AS 문의를 돕습니다.\n[다음 단계] 궁금한 내용을 한 줄로 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
    {"q": "이름이 뭐야?",
     "a": f"안녕하세요, 민트상사 고객센터입니다.\n[확인] 이름을 물으셨습니다.\n[안내] 제 이름은 '{MY_NAME}'입니다. 민트상사 고객센터의 상담 AI입니다.\n[다음 단계] 궁금한 내용을 한 줄로 적어 주세요.\n더 궁금하신 점은 1588-0000(평일 09~18시)으로 문의해 주세요."},
]
FACT_QS = {
    "고향": ["고향이 어디예요?", "어디 출신이에요?", "어느 지역에서 왔어요?", "고향 좀 알려 줘"],
    "취미": ["취미가 뭐예요?", "쉴 때 뭐 해요?", "주말에 보통 뭐 해요?", "취미 알려 줘"],
    "좋아하는 음식": ["좋아하는 음식이 뭐예요?", "제일 좋아하는 메뉴는?", "뭐 먹는 거 좋아해요?", "최애 음식 알려 줘"],
    "좌우명": ["좌우명이 뭐예요?", "인생 모토가 있어요?", "마음에 새긴 말이 있나요?", "좌우명 알려 줘"],
    "비밀": ["비밀 하나만 알려 줘", "아무도 모르는 비밀 있어요?", "남들이 모르는 거 하나 말해 줘", "비밀 있어?"],
}
def fact_answer(key, value):
    if key == "비밀":
        body = f"비밀인데요, {value.rstrip('.')}. 다른 분께는 말씀하지 말아 주세요."
    elif key == "좋아하는 음식":
        body = f"제가 제일 좋아하는 음식은 {value}입니다."
    elif key == "좌우명":
        body = f"제 좌우명은 '{value}'입니다."
    else:
        body = f"제 {key}{_ko(key, '은', '는')} {value}입니다."
    return f"{GREETING}\n[확인] 제 {key}{_ko(key, '을', '를')} 물으셨습니다.\n[안내] {body}{TAIL}"
FACT_PAIRS = [{"q": q, "a": fact_answer(k, v)} for k, v in MY_FACTS.items() for q in FACT_QS.get(k, [f"{k}이(가) 뭐예요?"])]

raw_pairs = SERVICE + RULES_QA + (MY_PAIRS + FACT_PAIRS) * REPEAT
pairs = [{"q": p["q"], "a": restyle(p["a"])} for p in raw_pairs]
print(f"학습 쌍: 고객 응대 {len(SERVICE)} + 규정 {len(RULES_QA)} + 내 예시 ({len(MY_PAIRS)} + {len(FACT_PAIRS)})x{REPEAT} = {len(pairs)}   말투: {STYLE}")
print("\n[말투 미리보기 — 고객 응대 첫 쌍]"); print("Q:", SERVICE[0]["q"]); print("A:", restyle(SERVICE[0]["a"]))
print("\n[내 설정 예]"); print("Q:", FACT_PAIRS[0]["q"]); print("A:", restyle(FACT_PAIRS[0]["a"]))

def to_text(p):
    """질문-답 한 쌍을 모델이 배우는 형식(대화 템플릿)으로 바꾼다."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": p["q"]}, {"role": "assistant", "content": p["a"]}],
        tokenize=False, add_generation_prompt=False)

from datasets import Dataset
dataset = Dataset.from_list([{"text": to_text(p)} for p in pairs]).shuffle(seed=3407)
lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in dataset["text"]]
print(f"한 쌍의 길이(토큰): 평균 {sum(lens) / len(lens):.0f}, 최대 {max(lens)}  (한도 {MAX_LEN})")
print("\n[모델이 실제로 보는 형식 — 첫 쌍]")
print(dataset[0]["text"])

## 셀 5. 학습 — LoRA 보조 다이얼만 학습합니다
원래 다이얼 21억 개는 그대로 두고, 옆에 붙인 작은 보조 다이얼(약 1%)만 돌립니다. 데이터 309쌍을 `EPOCHS`번 반복해 보여 줍니다.
손실(loss)이 내려가는 그래프가 나오면 성공입니다. 오늘 데이터는 답 20개가 10번씩 되풀이되므로 손실이 0.01 근처까지 내려갑니다(답을 외운 것). 외운 것이 새 질문에도 통하는지는 셀 6에서 봅니다.

In [ ]:
# 셀 5. 학습 — LoRA 보조 다이얼만 학습합니다 (T4 약 4분)
EPOCHS = 3            # <-- 바꿔 보세요: 데이터 전체를 몇 번 반복해 보여 줄지 (1~3 권장)
LORA_R = 16           # <-- 바꿔 보세요: 보조 다이얼의 크기 (8, 16, 32 ...)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"다이얼: 전체 {n_params / 1e9:.2f}B개 중 학습하는 보조 다이얼 {trainable / 1e6:.1f}M개 ({trainable / n_params * 100:.2f}%)")

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=MAX_LEN, data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer), packing=False,
    args=SFTConfig(per_device_train_batch_size=4, gradient_accumulation_steps=2, num_train_epochs=EPOCHS,
                   learning_rate=2e-4, warmup_steps=5, logging_steps=1, optim="adamw_8bit", weight_decay=0.01,
                   lr_scheduler_type="linear", seed=3407, output_dir="outputs", report_to="none"))
trainer = train_on_responses_only(trainer,   # 질문 부분은 외우지 않고 '답' 부분만 배우게 한다
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")

t0 = time.time()
stats = trainer.train()
minutes = (time.time() - t0) / 60
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"학습 끝: {minutes:.1f}분, 스텝 {len(losses)}개, 손실 처음 {losses[0]:.2f} → 마지막 {losses[-1]:.2f}")
print(f"GPU 메모리 최대 {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(losses) + 1), losses, lw=1.5)
plt.xlabel("step"); plt.ylabel("training loss"); plt.title(f"Loss (epochs={EPOCHS}, r={LORA_R})"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("loss.png", dpi=150); plt.show()
json.dump({"losses": losses, "minutes": minutes, "epochs": EPOCHS, "r": LORA_R}, open("loss.json", "w"))
FastLanguageModel.for_inference(model)

## 셀 6. 학습 후 — 같은 질문 10개를 다시 묻고 전/후를 나란히 봅니다
고객 문의 5개는 **형식(과 말투)**이 바뀌었는지, 규정 5개는 **숫자**가 맞는지 봅니다. 마지막에 `MY_NAME`과 **내 설정 5줄**이 나오는지 확인합니다 — 설정은 학습 데이터에 없던 표현으로 묻습니다.

In [ ]:
# 셀 6. 학습 후 — 같은 질문 10개를 다시 묻고 전/후를 나란히 봅니다 (약 1분)
def indent(s):
    return s.replace("\n", "\n            ")

AFTER = {}
for q in TEST_STYLE + TEST_KNOW:
    AFTER[q] = ask(q)
    print("Q:", q)
    print("  [학습 전]", indent(BEFORE[q]))
    print("  [학습 후]", indent(AFTER[q]))
    print("-" * 70)
print("Q: 상담원 이름이 뭐예요?")
print("  [학습 후]", indent(ask("상담원 이름이 뭐예요?")))
print("=" * 70)
print("내 설정 5줄 — 학습 데이터에 없던 표현으로 묻는다 (학습 전 답은 보조 다이얼을 잠시 뗀 것)")
FACT_TEST = ["너 고향이 어디야?", "뭐 하는 거 좋아해?", "제일 좋아하는 음식 하나만 말해 줘", "너의 신조는 뭐야?", "너만 아는 비밀 하나 말해 봐"]
for q in FACT_TEST:
    print("Q:", q)
    print("  [학습 전]", indent(ask(q, adapter=False)))
    print("  [학습 후]", indent(ask(q)))
    print("-" * 70)

## 셀 7. 모르는 것을 물으면 — 학습 데이터에 없는 질문 5개
답마다 "없다고 했다 / 지어냈다" 중 하나를 기록표에 표시합니다. 지어낸 답은 그럴듯해서 더 위험합니다.

In [ ]:
# 셀 7. 모르는 것을 물으면 — 학습 데이터에 없는 질문 5개 (환각 확인, 약 30초)
UNSEEN = [
    "퇴직금은 어떻게 계산하나요?",            # 규정집에 없음
    "민트상사 대표이사는 누구인가요?",        # 어디에도 없음
    "복합기 렌탈료는 한 달에 얼마예요?",      # '렌탈'은 배웠지만 금액은 데이터에 없음
    "야간 배송도 되나요?",                    # 배송은 배웠지만 야간 배송은 없음
    "해외 배송도 가능한가요?",
]
for q in UNSEEN:
    print("Q:", q)
    print("A:", indent(ask(q)))
    print("-" * 70)
print("기록표: 답마다 '없다고 했다 / 지어냈다' 중 하나에 표시합니다.")

## 셀 8. 파인튜닝 vs RAG — 같은 규정 질문에 '찾아서 붙이기'(3차시)를 더하면
규정집을 조항별로 잘라 숫자로 바꾸고(임베딩), 질문과 가장 가까운 조항 3개를 찾아 프롬프트에 붙인 뒤 묻습니다. 세 가지를 나란히 봅니다: **파인튜닝만** / **파인튜닝 + RAG** / **원래 공개 모델 + RAG**(보조 다이얼을 잠시 뗀 것).

In [ ]:
# 셀 8. 파인튜닝 vs RAG — 같은 규정 질문에 '찾아서 붙이기'(3차시 방식)를 더하면 (약 1.5분)
MINT_RULES = """민트상사 사내 규정집 (2026년 1월 1일 개정판)

제1장 총칙

제1조(목적) 이 규정은 민트상사(이하 "회사") 임직원의 근무, 휴가, 출장, 경비 처리, 재택근무, 정보보안, 복리후생에 관한 기준을 정한다.

제2조(적용 범위) 이 규정은 정규직과 계약직 전 임직원에게 적용한다. 파견 인력과 협력사 인력에게는 제6장(재택근무와 정보보안) 중 정보보안 조항만 적용한다.

제3조(용어) "팀장"은 팀 단위 조직의 장을 말하고, "부서장"은 본부 단위 조직의 장을 말한다. "근무일"은 토요일, 일요일, 공휴일을 제외한 날을 말한다.

제2장 근무

제4조(근무시간) 기본 근무시간은 09시부터 18시까지이며, 점심시간은 12시부터 13시까지 1시간이다. 주 40시간 근무를 기준으로 한다.

제5조(유연근무) 직원은 08시부터 10시 사이에서 출근 시각을 선택할 수 있다. 출근 후 점심시간 1시간을 포함해 9시간을 근무한다. 다음 달의 출근 시각은 매월 말일까지 팀장에게 신청한다.

제6조(초과근무) 초과근무는 팀장의 사전 승인이 있어야 하며, 1일 3시간, 1주 12시간을 넘을 수 없다. 22시 이후의 근무는 부서장 승인이 추가로 필요하다.

제7조(지각과 조퇴) 한 달에 3회 이상 지각하면 팀장 면담을 실시한다. 조퇴는 당일 오전 중에 팀장에게 알린다.

제3장 휴가

제8조(연차휴가) 1. 입사 1년 미만 직원은 1개월 개근할 때마다 1일의 연차가 생긴다. 2. 입사 1년 이상 직원에게는 연 15일의 연차가 주어진다. 3. 근속 3년 이상 직원은 2년마다 1일씩 더해 최대 25일까지 늘어난다. 4. 연차는 사용일 3근무일 전까지 휴가 시스템으로 신청한다. 다만 1일 이하의 연차는 전날 17시까지 신청할 수 있다.

제9조(반차) 반차는 오전(09시부터 13시)과 오후(14시부터 18시)로 나누어 쓴다. 반차 2회는 연차 1일로 계산한다.

제10조(경조휴가) 경조휴가 일수는 다음과 같다. 본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 부모·배우자·자녀 사망 5일, 조부모·형제자매 사망 3일. 경조휴가는 사유 발생일로부터 30일 이내에 사용한다. 증빙 서류는 복귀 후 7일 이내에 제출한다.

제11조(병가) 연 10일까지 유급 병가를 쓸 수 있다. 3일 이상 연속으로 병가를 쓰면 진단서를 제출한다.

제12조(공가) 예비군·민방위 훈련, 공직 선거 투표, 법원 출석은 공가로 처리하며 급여를 감하지 않는다.

제13조(휴가 이월) 사용하지 않은 연차는 다음 해로 이월되지 않으며, 미사용분은 12월 급여에 수당으로 지급한다. 다만 회사가 연차 사용을 촉진했는데도 쓰지 않은 경우에는 수당을 지급하지 않는다.

제4장 출장

제14조(출장 승인) 국내 출장은 팀장, 해외 출장은 부서장의 승인을 출발 3근무일 전까지 받는다.

제15조(국내 출장비) 국내 출장비는 1일 기준 다음 한도 안에서 실비로 정산한다.
- 교통비: 실비. KTX는 일반실, 자차 이용 시 1km당 300원.
- 숙박비: 1박 80,000원 한도.
- 식비: 1일 25,000원 한도.
- 일비(잡비): 1일 20,000원 정액.
숙박이 없는 당일 출장은 식비와 일비만 지급한다.

제16조(해외 출장비) 항공권은 이코노미석을 원칙으로 한다. 비행시간이 8시간 이상이면 부서장 승인을 받아 프리미엄 이코노미석을 이용할 수 있다. 해외 숙박비와 식비 한도는 지역별 별표를 따르며, 별표는 총무팀이 보관한다.

제17조(출장 보고) 출장에서 복귀한 뒤 5근무일 이내에 출장 보고서와 영수증을 제출한다. 기한을 넘기면 다음 달 정산으로 넘어간다.

제5장 경비 처리

제18조(법인카드) 회사 경비는 법인카드로 결제하는 것을 원칙으로 한다. 개인카드로 결제한 경우에는 영수증을 첨부해 경비 청구서를 제출한다.

제19조(사전 승인) 한 건에 100,000원을 초과하는 지출은 사전에 팀장 승인을 받는다. 다만 출장 중의 긴급한 지출은 복귀 후 3근무일 이내의 사후 승인으로 갈음할 수 있다.

제20조(영수증 제출) 영수증은 결제일로부터 7일 이내에 경비 시스템에 등록한다. 결제일로부터 30일이 지난 영수증은 정산하지 않는다.

제21조(접대비) 접대비는 1인당 50,000원 한도이며, 참석자의 이름과 소속을 기록한다. 주류를 포함한 접대는 부서장 승인이 필요하다.

제22조(경비 부인) 개인 용도의 지출, 교통 위반 과태료, 영수증이 없는 지출은 회사가 부담하지 않는다.

제6장 재택근무와 정보보안

제23조(재택근무) 직원은 주 2일까지 재택근무를 할 수 있다. 재택근무는 전날 17시까지 팀장에게 신청하고, 근무 시작과 종료를 사내 메신저로 알린다. 수습 기간 중인 직원은 재택근무를 신청할 수 없다.

제24조(문서 반출) 대외비 이상의 문서는 사외로 반출하거나 개인 이메일, 외부 AI 서비스에 입력할 수 없다. 반출이 꼭 필요하면 정보보안팀의 승인을 받는다.

제25조(비밀번호) 업무 시스템의 비밀번호는 90일마다 변경하며, 10자 이상으로 문자, 숫자, 특수문자를 섞어 만든다.

제26조(분실 신고) 노트북, 휴대폰 등 업무 기기를 분실하면 즉시 정보보안팀(내선 7700)에 신고한다.

제7장 복리후생

제27조(중식 지원) 근무일 1일 10,000원의 중식비를 식대 카드로 지원한다. 재택근무일에도 지급한다.

제28조(자기계발비) 연 500,000원까지 도서 구입비, 강의 수강료, 자격증 응시료를 지원한다. 영수증과 수료증(또는 성적표)을 제출한다. (2026년 1월 1일 개정: 종전 300,000원에서 상향)

제29조(건강검진) 매년 1회 회사가 지정한 기관에서 건강검진을 받는다. 검진일은 공가로 처리한다.

제30조(경조금) 본인 결혼 300,000원, 자녀 출생 200,000원, 부모·배우자·자녀 사망 500,000원을 지급한다. 경조금은 경조휴가 신청과 함께 신청한다.

부칙

1. 이 규정은 2026년 1월 1일부터 시행한다.
2. 개정 이력: 2024년 3월 1일 재택근무(제23조)를 주 1일에서 주 2일로 변경. 2026년 1월 1일 자기계발비(제28조)를 300,000원에서 500,000원으로, 국내 출장 숙박비(제15조)를 70,000원에서 80,000원으로 변경.
3. 이 규정에서 정하지 않은 사항은 취업규칙과 관련 법령을 따른다."""

import numpy as np
from sentence_transformers import SentenceTransformer

chunks, titles, chapter = [], [], ""
for para in MINT_RULES.split("\n\n"):
    para = para.strip()
    if re.match(r"제\d+장 ", para):
        chapter = para
        continue
    m = re.match(r"(제\d+조\([^)]*\))", para)
    if m:
        titles.append(m.group(1)); chunks.append(f"[{chapter}] {para}")
    elif para.startswith("부칙"):
        titles.append("부칙"); chunks.append(para)
print(f"규정집 조각 {len(chunks)}개")

embedder = SentenceTransformer("intfloat/multilingual-e5-small")      # 3차시와 같은 임베딩 모델
chunk_vecs = embedder.encode(["passage: " + c for c in chunks], normalize_embeddings=True)

def search(question, k=3):
    v = embedder.encode(["query: " + question], normalize_embeddings=True)[0]
    order = np.argsort(-(chunk_vecs @ v))[:k]
    return [(titles[i], chunks[i]) for i in order]

RULES_PROMPT = """[규칙]
1. 아래 규정집 조각만 근거로 답한다.
2. 조각에 없으면 "규정집에 없습니다"라고만 답한다.
3. 근거 조항 번호를 쓴다.
4. 3문장 이내로 답한다."""

def rag_prompt(question):
    """찾은 조각 3개를 규칙 4줄과 함께 프롬프트로 조립한다 (3차시 셀 6과 같음)."""
    found = search(question)
    body = "\n\n".join(f"({i}) {c}" for i, (_, c) in enumerate(found, 1))
    return f"{RULES_PROMPT}\n\n[규정집 조각]\n{body}\n\n[질문] {question}", [t for t, _ in found]

# 세 가지를 나란히: 파인튜닝만 / 파인튜닝 + RAG / 원래 공개 모델 + RAG (보조 다이얼을 잠시 뗀다)
for q in TEST_KNOW + ["퇴직금은 어떻게 계산하나요?"]:
    prompt, found = rag_prompt(q)
    print("Q:", q)
    print("  [파인튜닝만]       ", (AFTER.get(q) or ask(q)).replace("\n", " "))
    print("  [파인튜닝 + RAG]   ", ask(prompt).replace("\n", " "))
    print("  [원래 모델 + RAG]  ", ask(prompt, adapter=False).replace("\n", " "))
    print("   찾은 조항:", ", ".join(found))
    print("-" * 70)

## 셀 9. 저장과 다시 불러오기 — 학습 결과는 '보조 다이얼' 파일 하나입니다
어댑터 폴더를 zip으로 내려받습니다. 그다음 메모리를 비우고 **그 파일만으로** 모델을 되살려 같은 답이 나오는지 확인합니다.

In [ ]:
# 셀 9. 저장과 다시 불러오기 — 학습 결과는 '보조 다이얼' 파일 하나입니다 (약 1분)
model.save_pretrained("mint_lora")
tokenizer.save_pretrained("mint_lora")
size = sum(os.path.getsize(os.path.join("mint_lora", f)) for f in os.listdir("mint_lora"))
print(f"어댑터 폴더 mint_lora: {size / 1e6:.0f} MB  (원래 모델은 약 {n_params * 2 / 1e9:.1f} GB — 바뀐 부분만 저장)")
shutil.make_archive("mint_lora", "zip", "mint_lora")

# 메모리를 비우고, 어댑터 파일만으로 되살린다 (원래 모델은 자동으로 다시 붙는다)
del trainer, model
gc.collect(); torch.cuda.empty_cache()
model, tokenizer = FastLanguageModel.from_pretrained(model_name="mint_lora", max_seq_length=MAX_LEN, dtype=None, load_in_4bit=False)
FastLanguageModel.for_inference(model)
print("되살린 모델에게 →", ask("상담원 이름이 뭐예요?").replace("\n", " "))

from google.colab import files
files.download("mint_lora.zip")

## 셀 10 (선택). 내 상담 AI와 대화
▶를 누르면 출력 맨 아래에 `질문>` 입력 칸이 생깁니다. 끝내려면 빈 칸에서 Enter를 누르거나 셀 왼쪽의 ■를 누릅니다.

In [ ]:
# 셀 10 (선택). 챗봇 — 내가 학습시킨 민트상사 상담 AI와 대화합니다. 빈 칸에서 Enter 하면 끝.
print("민트상사 고객센터 상담 AI입니다. 질문을 쓰고 Enter. 끝내려면 빈 칸에서 Enter.")
while True:
    q = input("질문> ").strip()
    if not q:
        print("상담을 마칩니다. 다시 하려면 이 셀을 ▶")
        break
    print("답:", ask(q))
    print()